In [1]:
import pandas as pd

fixed_entries=[
{"question":"what is the annual fee","answer":"The annual fee is Rs 500.","keywords":"fee cost price charge","category":"billing"},
{"question":"how to reset password","answer":"Go to Settings > Reset Password.","keywords":"password reset login","category":"account"},
{"question":"what are your working hours","answer":"We are open 9 AM to 5 PM.","keywords":"hours timing open time","category":"general"},
{"question":"how can i pay the fee","answer":"You can pay via UPI, card, or net banking.","keywords":"pay payment upi fee","category":"billing"}
]

personalized_entry_1={
"question":"how do i update my registered mobile number",
"answer":"Go to Account Settings and update your registered mobile number.",
"keywords":"mobile number update",
"category":"account"
}

personalized_entry_2={
"question":"how can i change my registered email",
"answer":"Go to Account Settings and change your registered email address.",
"keywords":"email change account",
"category":"account"
}

df=pd.DataFrame(fixed_entries+[personalized_entry_1,personalized_entry_2])
print(df)

                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5         how can i change my registered email   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4  Go to Account Settings and update your registe...    mobile number update   
5  Go to Account Settings and change your registe...    email change account   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  account  
5  account

In [4]:
def score_query(query,df):
    query_words=set(query.lower().split())
    results=[]

    for index,row in df.iterrows():
        text=row["question"]+" "+row["keywords"]
        text_words=set(text.lower().split())
        score=len(query_words.intersection(text_words))

        if score>0:
            results.append({
                "question":row["question"],
                "answer":row["answer"],
                "keywords":row["keywords"],
                "category":row["category"],
                "score":score
            })

    return pd.DataFrame(sorted(results,key=lambda x:x["score"],reverse=True))

query=input("Enter your query: ")
result=score_query(query,df)

print(result if not result.empty else "No matching FAQ found.")

Enter your query:  what is the annual fee


                      question                                      answer  \
0       what is the annual fee                   The annual fee is Rs 500.   
1        how can i pay the fee  You can pay via UPI, card, or net banking.   
2  what are your working hours                   We are open 9 AM to 5 PM.   

                 keywords category  score  
0   fee cost price charge  billing      5  
1     pay payment upi fee  billing      2  
2  hours timing open time  general      1  


In [5]:
def same_category(category_name,df):
    return df[df["category"]==category_name]

category_name="account"
result=same_category(category_name,df)

print(result)

                                      question  \
1                        how to reset password   
4  how do i update my registered mobile number   
5         how can i change my registered email   

                                              answer              keywords  \
1                   Go to Settings > Reset Password.  password reset login   
4  Go to Account Settings and update your registe...  mobile number update   
5  Go to Account Settings and change your registe...  email change account   

  category  
1  account  
4  account  
5  account  


In [6]:
print(df.loc[0])

new_keyword=input("Enter a new keyword: ")

df.loc[0,"keywords"]=df.loc[0,"keywords"]+" "+new_keyword

df.to_csv("1024170214_faq_data.csv",index=False)

print(df)

question       what is the annual fee
answer      The annual fee is Rs 500.
keywords        fee cost price charge
category                      billing
Name: 0, dtype: object


Enter a new keyword:  discount


                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5         how can i change my registered email   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Account Settings and update your registe...   
5  Go to Account Settings and change your registe...   

                         keywords category  
0  fee cost price charge discount  billing  
1            password reset login  account  
2          hours timing open time  general  
3             pay payment upi fee  billing  
4            mobile number upda

In [7]:
category_counts=df.groupby("category").size()
print(category_counts)

category
account    3
billing    2
general    1
dtype: int64


In [8]:
def score_query_with_ties(query,df):
    query_words=set(query.lower().split())
    results=[]
    for index,row in df.iterrows():
        text=row["question"]+" "+row["keywords"]
        text_words=set(text.lower().split())
        score=len(query_words.intersection(text_words))
        if score>0:
            results.append({
                "question":row["question"],
                "answer":row["answer"],
                "keywords":row["keywords"],
                "category":row["category"],
                "score":score
            })

    if not results:
        return pd.DataFrame()
    result_df=pd.DataFrame(results)
    highest_score=result_df["score"].max()
    return result_df[result_df["score"]==highest_score]
print("Tie query:")
print(score_query_with_ties("fee",df))
print("\nNon-tie query:")
print(score_query_with_ties("password",df))

Tie query:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
1   how can i pay the fee  You can pay via UPI, card, or net banking.   

                         keywords category  score  
0  fee cost price charge discount  billing      1  
1             pay payment upi fee  billing      1  

Non-tie query:
                question                            answer  \
0  how to reset password  Go to Settings > Reset Password.   

               keywords category  score  
0  password reset login  account      1  
